In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')

!pip install -q kagglehub segmentation-models-pytorch albumentations timm


Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 13.4 MB/s eta 0:00:00


In [2]:
# Define paths in Google Drive
BASE_DIR = "/content/drive/MyDrive/EchoNet_Colab"
RAW_DIR = f"{BASE_DIR}/raw"
DATA_DIR = f"{BASE_DIR}/dataset"
IMG_DIR = f"{DATA_DIR}/images"
MASK_DIR = f"{DATA_DIR}/masks"
CKPT_DIR = f"{BASE_DIR}/checkpoints"

# Create directories
for d in [RAW_DIR, IMG_DIR, MASK_DIR, CKPT_DIR]:
    os.makedirs(d, exist_ok=True)
print("✅ Directories ready in Google Drive.")


✅ Directories ready in Google Drive.


In [3]:
import kagglehub
import shutil
import cv2
import pandas as pd
import numpy as np
import glob
from tqdm import tqdm

dataset_path = f"{RAW_DIR}/EchoNet-Dynamic"

# Download if not exists
if not os.path.exists(dataset_path):
    print("Downloading dataset...")
    dl_path = kagglehub.dataset_download("mahnurrahman/echonet-dynamic")
    shutil.copytree(dl_path, dataset_path, dirs_exist_ok=True)

# Generate dataset (Skip if already generated > 1000 images)
if len(glob.glob(f"{IMG_DIR}/*.png")) < 1000:
    print("Processing videos and generating masks...")
    df = pd.read_csv(f"{dataset_path}/EchoNet-Dynamic/VolumeTracings.csv")
    grouped = df.groupby(["FileName", "Frame"])

    SAVE_SIZE = 256
    counter = 0

    for (fname, frame_num), rows in tqdm(grouped):
        v_path = f"{dataset_path}/EchoNet-Dynamic/Videos/{fname}"
        if not os.path.exists(v_path): continue

        cap = cv2.VideoCapture(v_path)
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_num)
        ret, frame = cap.read()
        cap.release()

        if not ret: continue

        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        mask = np.zeros(frame.shape, dtype=np.uint8)

        # CORRECTED MASK GENERATION: using both X1,Y1 and X2,Y2
        p1, p2 = [], []
        for _, r in rows.iterrows():
            p1.append([int(r["X1"]), int(r["Y1"])])
            p2.append([int(r["X2"]), int(r["Y2"])])

        points = np.array(p1 + p2[::-1], dtype=np.int32)
        if len(points) > 5:
            cv2.fillPoly(mask, [points], 255)

            frame = cv2.resize(frame, (SAVE_SIZE, SAVE_SIZE))
            mask = cv2.resize(mask, (SAVE_SIZE, SAVE_SIZE))

            cv2.imwrite(f"{IMG_DIR}/{counter}.png", frame)
            cv2.imwrite(f"{MASK_DIR}/{counter}.png", mask)
            counter += 1
print("✅ Dataset and Masks are ready!")


100%|██████████| 6.56G/6.56G [05:55<00:00, 19.8MB/s]

Extracting files...


Processing videos and generating masks...


100%|██████████| 20050/20050 [13:58<00:00, 23.90it/s]

✅ Dataset and Masks are ready!


In [4]:
import torch
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split

images_list = sorted(os.listdir(IMG_DIR))
# random_state=42 ensures the train/val split is exactly the same across different Colab accounts!
train_imgs, valid_imgs = train_test_split(images_list, test_size=0.1, random_state=42)

train_aug = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.ElasticTransform(p=0.2),
    A.GridDistortion(p=0.2),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.5),
    A.Normalize(mean=(0.5,), std=(0.5,)),
    ToTensorV2()
])
valid_aug = A.Compose([A.Normalize(mean=(0.5,), std=(0.5,)), ToTensorV2()])

class EchoData(Dataset):
    def __init__(self, imgs, transform):
        self.imgs = imgs
        self.transform = transform

    def __len__(self): return len(self.imgs)

    def __getitem__(self, idx):
        name = self.imgs[idx]
        img = cv2.imread(f"{IMG_DIR}/{name}", cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(f"{MASK_DIR}/{name}", cv2.IMREAD_GRAYSCALE)
        aug = self.transform(image=img, mask=mask)
        return aug["image"].float(), (aug["mask"].unsqueeze(0).float() / 255.0)

train_loader = DataLoader(EchoData(train_imgs, train_aug), batch_size=16, shuffle=True, num_workers=2)
valid_loader = DataLoader(EchoData(valid_imgs, valid_aug), batch_size=16, shuffle=False, num_workers=2)


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [5]:
import segmentation_models_pytorch as smp
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = smp.UnetPlusPlus(
    encoder_name="timm-efficientnet-b4",
    encoder_weights="noisy-student",
    in_channels=1,
    classes=1
).to(device)

dice_loss = smp.losses.DiceLoss(mode="binary")
bce_loss = nn.BCEWithLogitsLoss()
def criterion(pred, target): return 0.5 * dice_loss(pred, target) + 0.5 * bce_loss(pred, target)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
scaler = torch.amp.GradScaler('cuda')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/77.9M [00:00<?, ?B/s]

In [6]:
LATEST_CKPT = f"{CKPT_DIR}/latest.pth"
BEST_CKPT = f"{CKPT_DIR}/best.pth"

start_epoch, best_dice, patience = 0, 0.0, 0
MAX_EPOCHS, EARLY_STOP = 100, 10

# ---- AUTO RESUME LOGIC ----
if os.path.exists(LATEST_CKPT):
    print("🔄 Existing checkpoint found! Resuming training...")
    ckpt = torch.load(LATEST_CKPT, map_location=device)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    scaler.load_state_dict(ckpt['scaler'])
    start_epoch = ckpt['epoch']
    best_dice = ckpt['best_dice']
    patience = ckpt['patience']
    print(f"✅ Resumed successfully from Epoch {start_epoch} | Best Dice so far: {best_dice:.4f}")
else:
    print("🚀 Starting training from scratch...")
# ---------------------------

def calc_dice(p, t):
    p = (p > 0.5).float()
    return (2*(p*t).sum() + 1e-6) / (p.sum() + t.sum() + 1e-6)

for epoch in range(start_epoch, MAX_EPOCHS):
    print(f"\n--- Epoch {epoch+1}/{MAX_EPOCHS} ---")

    model.train()
    train_loss = 0
    for imgs, masks in tqdm(train_loader, desc="Train"):
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda'):
            preds = model(imgs)
            loss = criterion(preds, masks)

        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()

    model.eval()
    val_loss, val_dice = 0, 0
    with torch.no_grad():
        for imgs, masks in tqdm(valid_loader, desc="Valid"):
            imgs, masks = imgs.to(device), masks.to(device)
            preds = model(imgs)
            val_dice += calc_dice(torch.sigmoid(preds), masks).item()

    val_dice /= len(valid_loader)
    scheduler.step(val_dice)
    print(f"Train Loss: {train_loss/len(train_loader):.4f} | Val Dice: {val_dice:.4f}")

    if val_dice > best_dice:
        best_dice, patience = val_dice, 0
        torch.save(model.state_dict(), BEST_CKPT)
        print("⭐ New Best Model Saved!")
    else:
        patience += 1

    # Save checkpoint immediately to Google Drive (allows safe switching)
    torch.save({
        'epoch': epoch + 1,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'scaler': scaler.state_dict(),
        'best_dice': best_dice,
        'patience': patience
    }, LATEST_CKPT)

    if patience >= EARLY_STOP:
        print("🛑 Early Stopping triggered. Training finished.")
        break


🚀 Starting training from scratch...

--- Epoch 1/100 ---


Valid: 100%|██████████| 126/126 [00:40<00:00,  3.07it/s]


Train Loss: 0.1141 | Val Dice: 0.9128
⭐ New Best Model Saved!

--- Epoch 2/100 ---


Valid: 100%|██████████| 126/126 [00:40<00:00,  3.09it/s]


Train Loss: 0.0713 | Val Dice: 0.9178
⭐ New Best Model Saved!

--- Epoch 3/100 ---


...

Train Loss: 0.0618 | Val Dice: 0.9193
⭐ New Best Model Saved!

--- Epoch 10/100 ---


Valid: 100%|██████████| 126/126 [00:28<00:00,  4.40it/s]


Train Loss: 0.0604 | Val Dice: 0.9182

--- Epoch 11/100 ---


Train:   7%|▋         | 83/1128 [00:40<08:34,  2.03it/s]


KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt

def visualize_predictions():
    if not os.path.exists(BEST_CKPT):
        print("No trained model found yet!")
        return

    model.load_state_dict(torch.load(BEST_CKPT, map_location=device))
    model.eval()

    imgs, masks = next(iter(valid_loader))
    imgs = imgs.to(device)

    with torch.no_grad():
        preds = torch.sigmoid(model(imgs)).cpu().numpy()

    imgs = imgs.cpu().numpy()
    masks = masks.cpu().numpy()

    fig, axes = plt.subplots(3, 3, figsize=(10, 10))
    for i in range(3):
        img_show = imgs[i][0] * 0.5 + 0.5

        axes[i, 0].imshow(img_show, cmap='gray')
        axes[i, 0].set_title("Original")
        axes[i, 0].axis('off')

        axes[i, 1].imshow(masks[i][0], cmap='gray')
        axes[i, 1].set_title("Ground Truth Mask")
        axes[i, 1].axis('off')

        axes[i, 2].imshow(img_show, cmap='gray')
        axes[i, 2].imshow(preds[i][0] > 0.5, cmap='Reds', alpha=0.5)
        axes[i, 2].set_title("Prediction Overlay")
        axes[i, 2].axis('off')

    plt.tight_layout()
    plt.show()

visualize_predictions()
